<a href="https://colab.research.google.com/github/urcraft/demi_api_lecture_notebooks/blob/main/MCP_Todo_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From REST API to AI-Ready Tool using MCP

**Course:** Digital Enterprise Management: Infrastructure Perspective — Lecture IV  
**Instructor:** Unnikrishnan R., Assistant Professor, BTECH, Aarhus University

---

## What is this about?

In the previous exercise, you built a **Todo REST API** using Flask and tested it with Postman using standard HTTP requests (GET, POST, PUT, DELETE).

Now you’ll take the **same todo data** and wrap it as an **MCP server** — making it discoverable and callable by AI agents.

By the end of this notebook you will have:
1. Connected to a **public MCP server** from Postman (to see what MCP looks like from the client side)
2. Built your own **MCP server** wrapping the same todo data from the Flask exercise
3. Exposed it to the internet via **ngrok** (same pattern you already know)
4. Connected to **your own MCP server** from Postman’s MCP client

The key insight: **same data, same tool (Postman), different protocol.** REST is for human developers reading docs. MCP is for AI agents discovering tools automatically.

---

## Step 1: Install Dependencies

We need two packages:
- **FastMCP** — the Python framework for building MCP servers
- **pyngrok** — to expose our local server to the internet (you used this in the Flask exercise too)

In [ ]:
# Run this cell first — only needs to run once
!pip install fastmcp pyngrok -q

# Verify installation
import fastmcp
print(f"✅ FastMCP version: {fastmcp.__version__}")

## Step 2: Build Your Own MCP Server — Wrapping the Todo Data

Now let’s build an MCP server using the **exact same todo data** from your Flask exercise.

In Flask, you defined HTTP endpoints like `@app.route('/api/todos', methods=['GET'])`.  
In MCP, you define **tools** with `@mcp.tool` — functions that an AI agent can discover and call.

Compare them side by side:

| Flask (REST) | FastMCP (MCP) |
|---|---|
| `@app.route('/api/todos', methods=['GET'])` | `@mcp.tool` |
| Client sends HTTP request to a URL | Client calls a tool by name |
| Client needs to know the URL and method | Client discovers tools automatically |
| Returns JSON via `jsonify()` | Returns data directly from the function |
| Docs written by you (or forgotten) | Docs = the function’s docstring + type hints |

> **📖 Read the code and comments carefully — the pattern is always the same:**
> 1. Create a server with `FastMCP("name")`
> 2. Decorate a function with `@mcp.tool`
> 3. Add type hints and a clear docstring
> 4. The function IS the tool

In [ ]:
from fastmcp import FastMCP

# ============================================================
# CREATE THE MCP SERVER
# ============================================================
# The name is what AI agents see when they connect.
# Think of it like a business card for your server.

mcp = FastMCP("Todo Server")

# ============================================================
# DATA — identical to what you used in Flask
# ============================================================
# Same in-memory list. Same structure. Same starting data.
# The only difference is HOW we expose it.

todos = [
    {"id": 1, "title": "Learn REST APIs", "completed": False},
    {"id": 2, "title": "Build a Flask API", "completed": False},
    {"id": 3, "title": "Share with classmates", "completed": False}
]

# ============================================================
# TOOL 1: Get all todos
# ============================================================
# In Flask this was:  @app.route('/api/todos', methods=['GET'])
# In MCP it's just a decorated function.
#
# The docstring is CRITICAL — it's what an AI agent reads to
# decide whether to call this tool. Write it for a machine.

@mcp.tool
def get_all_todos() -> list[dict]:
    """Retrieve all todo items. Returns a list of todos, each with id, title, and completed status."""
    return todos

# ============================================================
# TOOL 2: Get a single todo by ID
# ============================================================
# In Flask:  @app.route('/api/todos/<int:todo_id>', methods=['GET'])
# In MCP:    the parameter is just a function argument with a type hint

@mcp.tool
def get_todo(todo_id: int) -> dict:
    """Retrieve a specific todo by its ID. Returns the todo if found, or an error message."""
    for todo in todos:
        if todo["id"] == todo_id:
            return todo
    return {"error": f"Todo {todo_id} not found"}

# ============================================================
# TOOL 3: Create a new todo
# ============================================================
# In Flask:  @app.route('/api/todos', methods=['POST'])
#            data came from request.json['title']
# In MCP:    data comes as function parameters — cleaner!

@mcp.tool
def add_todo(title: str) -> dict:
    """Create a new todo item. Provide a title; the todo starts as not completed."""
    new_id = max(t["id"] for t in todos) + 1
    todo = {"id": new_id, "title": title, "completed": False}
    todos.append(todo)
    return todo

# ============================================================
# TOOL 4: Mark a todo as completed
# ============================================================
# In Flask:  @app.route('/api/todos/<int:todo_id>', methods=['PATCH'])
#            with body {"completed": true}
# In MCP:    just a function call with the ID

@mcp.tool
def complete_todo(todo_id: int) -> dict:
    """Mark a specific todo as completed. Provide the todo ID."""
    for todo in todos:
        if todo["id"] == todo_id:
            todo["completed"] = True
            return todo
    return {"error": f"Todo {todo_id} not found"}

# ============================================================
# TOOL 5: Delete a todo
# ============================================================
# In Flask:  @app.route('/api/todos/<int:todo_id>', methods=['DELETE'])
# In MCP:    same idea, different interface

@mcp.tool
def delete_todo(todo_id: int) -> dict:
    """Delete a todo by its ID. Returns success message or error if not found."""
    for todo in todos:
        if todo["id"] == todo_id:
            todos.remove(todo)
            return {"message": f"Todo {todo_id} deleted successfully"}
    return {"error": f"Todo {todo_id} not found"}

print("✅ MCP Server defined with 5 tools:")
print("   - get_all_todos")
print("   - get_todo")
print("   - add_todo")
print("   - complete_todo")
print("   - delete_todo")
print()
print("Run the next cell to start the server and expose it via ngrok.")

## Step 4: Quick Test — Simulating an AI Agent (In-Memory)

Before we go to the network, let’s verify the server works. FastMCP has a built-in `Client` that can connect to a server in memory — no network needed.

This is exactly what an AI agent does:
1. **Discover** — “What tools are available?”
2. **Understand** — “What does each tool do? What parameters does it need?”
3. **Call** — “I’ll call `add_todo` with title ‘Test MCP’”

In [ ]:
import asyncio
from fastmcp import Client

async def test_todo_server():
    # Connect to our server in memory (no network needed)
    client = Client(mcp)

    async with client:
        # -- STEP 1: Discover available tools --
        tools = await client.list_tools()
        print("=" * 60)
        print("🔍 AI AGENT: What tools are available?")
        print("=" * 60)
        for tool in tools:
            print(f"  🛠️  {tool.name}: {tool.description}")
        print()

        # -- STEP 2: Call get_all_todos --
        print("📡 AI AGENT: Calling get_all_todos()...")
        result = await client.call_tool("get_all_todos", {})
        print(f"   Response: {result}")
        print()

        # -- STEP 3: Add a new todo --
        print("📡 AI AGENT: Calling add_todo(title='Test MCP from AI')...")
        result = await client.call_tool("add_todo", {"title": "Test MCP from AI"})
        print(f"   Response: {result}")
        print()

        # -- STEP 4: Complete a todo --
        print("📡 AI AGENT: Calling complete_todo(todo_id=1)...")
        result = await client.call_tool("complete_todo", {"todo_id": 1})
        print(f"   Response: {result}")
        print()

        # -- STEP 5: Check all todos again --
        print("📡 AI AGENT: Calling get_all_todos() again...")
        result = await client.call_tool("get_all_todos", {})
        print(f"   Response: {result}")

# Run the test
await test_todo_server()

---

## Step 5: Expose Your MCP Server via ngrok

Just like in the Flask exercise, we’ll use ngrok to give our local server a public URL.

The difference: in Flask, ngrok exposed an **HTTP endpoint**.  
Now, ngrok exposes an **MCP endpoint** (using SSE transport) that any MCP client can connect to.

### You’ll need your ngrok auth token
Same one you used for the Flask exercise. Get it from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

> ⚠️ **Important:** This cell starts the server and **keeps running**. You’ll see the ngrok URL printed — copy it and use it in Step 6. To stop the server, interrupt the cell (stop button).

In [ ]:
import threading
import getpass
from pyngrok import ngrok

# ============================================================
# STEP A: Set up ngrok (same pattern as Flask exercise)
# ============================================================
auth_token = getpass.getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(auth_token)

# The MCP server will run on port 8000
MCP_PORT = 8000

# Create the ngrok tunnel
public_url = ngrok.connect(MCP_PORT).public_url

print()
print("=" * 60)
print("✅ Your MCP server is accessible at:")
print(f"   {public_url}/sse")
print("=" * 60)
print()
print("📋 Copy the URL above and use it in Postman (Step 6)")
print()
print("To connect from Postman:")
print("  1. Open Postman → New → MCP")
print(f"  2. Paste: {public_url}/sse")
print("  3. Click Connect → Load Capabilities")
print("  4. Browse Tools → pick one → Run")
print()
print("🔴 Keep this cell running! Press the stop button when done.")
print("=" * 60)

# ============================================================
# STEP B: Start the MCP server with SSE transport
# ============================================================
# SSE (Server-Sent Events) is the transport that Postman and
# other MCP clients use to connect over HTTP.
#
# This is the equivalent of app.run() in Flask — it starts
# the server and keeps it running.

# ============================================================
# STEP B: Start the MCP server in a background thread
# ============================================================
# Colab already runs an asyncio event loop, so we can't call
# mcp.run() directly. We start it in a separate thread instead.

import uvicorn

server_thread = threading.Thread(
    target=mcp.run,
    kwargs={"transport": "sse", "host": "0.0.0.0", "port": MCP_PORT},
    daemon=True
)
server_thread.start()

import time
time.sleep(2)  # Give the server a moment to start
print("\n🟢 MCP server is running in the background!")
print(f"   Connect from Postman: {public_url}/sse")

---

## Step 6: Connect from Postman — Test Your MCP Server

Now the fun part. Open Postman and connect to **your own MCP server** — the one you just built.

### Instructions:

1. **Open Postman** and click **New** (or press Ctrl+N / Cmd+N)
2. Choose **MCP** from the request type dropdown
3. Paste your server URL (from Step 5): `https://xxxx-xx-xx.ngrok-free.app/sse`
4. Click **Connect**
5. Click **Load Capabilities**
6. You should see your 5 tools listed under the **Tools** tab:
   - `get_all_todos`
   - `get_todo`
   - `add_todo`
   - `complete_todo`
   - `delete_todo`

### Try these:

| # | Action | What to do in Postman |
|---|--------|----------------------|
| 1 | List all todos | Select `get_all_todos` → Run |
| 2 | Add a todo | Select `add_todo` → set `title` to your name → Run |
| 3 | Complete a todo | Select `complete_todo` → set `todo_id` to `1` → Run |
| 4 | Check the result | Select `get_all_todos` again → Run → see the changes |
| 5 | Delete a todo | Select `delete_todo` → set `todo_id` to `3` → Run |
| 6 | Edge case | Select `get_todo` → set `todo_id` to `999` → what happens? |

### 💡 Reflection

Compare this experience to the Flask exercise:
- In Flask, you had to know the URL path (`/api/todos`), the HTTP method (`GET`), and the request body format.
- In MCP, Postman **discovered** the tools, showed you the parameters, and let you call them directly.
- You didn’t write a single URL. The **tool described itself**.

Now imagine an AI agent doing this instead of you clicking buttons in Postman. That’s the MCP value proposition for enterprise infrastructure.

---

## Step 7: Side-by-Side Comparison — Flask vs MCP

You’ve now built the same application two ways. Here’s what changed and what didn’t:

### What stayed the same
- The **data** (same todo list, same structure)
- The **logic** (same operations: list, get, create, complete, delete)
- The **exposure tool** (ngrok to make it internet-accessible)
- The **client** (Postman — it speaks both REST and MCP!)

### What changed

| Aspect | Flask (REST) | FastMCP (MCP) |
|--------|-------------|---------------|
| **How it’s found** | Developer reads API docs | Client auto-discovers tools |
| **How it’s called** | HTTP method + URL + headers + body | Tool name + typed parameters |
| **Who uses it** | Human developers (or their code) | AI agents (or human developers) |
| **Documentation** | Separate (Swagger, README, /docs) | Built into the tool (docstrings + types) |
| **Protocol** | HTTP/REST | MCP (over SSE or HTTP) |
| **Error handling** | Status codes (404, 400, etc.) | Return values from the function |

### The key business insight

REST APIs are designed for **developers who read documentation**.  
MCP tools are designed for **AI agents that discover capabilities automatically**.

In an enterprise with hundreds of internal APIs, MCP means you don’t need a custom integration for every AI tool. You wrap your APIs once, and any MCP-compatible AI agent can find and use them.

---

## Step 8: (Demo) Connect an AI Agent to Your Server

This step is a **live demo by the instructor** — you don’t need to set this up yourself.

We’ll take one student group’s ngrok URL and connect it to an AI client:

### Option A: Claude Desktop
Claude Desktop can connect to MCP servers. The instructor adds your server URL to Claude’s config:
```json
{
  "mcpServers": {
    "student-todo-server": {
      "command": "npx",
      "args": ["-y", "mcp-remote", "https://xxxx.ngrok-free.app/sse"]
    }
  }
}
```
Then asks Claude: *“What todos are on my list? Add one called ‘Ace the exam’ and mark the first one as done.”*  
Claude discovers the tools, calls them, and responds in natural language.

### Option B: MCP Inspector
A debugging tool that shows exactly what the AI agent sees:
```bash
npx @modelcontextprotocol/inspector
```
Connect to the student’s URL, browse tools, call them, see raw request/response.

### Option C: GitHub Copilot (VS Code)
Copilot supports MCP servers as tool sources. The instructor adds the student’s server in VS Code settings and asks Copilot to interact with the todos.

### What to watch for:
- The AI **discovers** the tools without being told what they are
- The AI reads the **docstrings** to understand what each tool does
- The AI **chains** tool calls together (e.g., list → find → complete)
- **Your API is now callable by AI.** That’s the full arc.

---

## 🌍 Real-World MCP in Action

MCP isn’t just a classroom exercise. Real APIs are being wrapped as MCP servers right now:

### Danish examples
[Mikkel Krogsholm’s Agent Skills](https://github.com/mikkelkrogsholm/skills) — real Danish APIs wrapped for AI agents:
- **Salling Group** — food waste data, store information
- **Jobindex** — job search across Danish job listings
- **Boliga** — Danish property market data
- **PubMed / medRxiv** — medical research papers

### Enterprise tools with MCP servers
- **Postman** — the tool you used today has its own MCP server (`mcp.postman.com`)
- **Slack, Google Drive, Salesforce, Jira** — all have MCP server implementations
- **GitHub** — MCP server for repository management, issues, PRs

### The business pattern
Company has internal APIs → wraps them as MCP servers → AI agents across the organization can discover and use them → no custom integration per AI tool.

---

## 📝 Reflection Questions

Think about these as you finish up:

1. **Your Flask API vs your MCP server** — which was easier to understand as a *consumer*? Which required less prior knowledge to use?

2. **Enterprise scenario** — pick a business you know (your employer, a case study from another course). What internal tools or data would be most valuable to expose as MCP servers? Think: what do people repeatedly ask IT for help with?

3. **Documentation quality** — in MCP, the docstring IS the documentation. How does this change the way you’d write function descriptions compared to traditional API docs?

4. **Security implications** — if an AI agent can discover and call any tool on an MCP server, what guardrails would an enterprise need?

---

## 📚 Quick Reference

### MCP Tool Pattern
```python
from fastmcp import FastMCP

mcp = FastMCP("Server Name")

@mcp.tool
def my_tool(param1: str, param2: int) -> dict:
    """Clear description for AI agents. Include what it does and what it returns."""
    # Your logic here
    return {"result": "value"}

# Run with SSE transport for network access
mcp.run(transport="sse", host="0.0.0.0", port=8000)
```

### Postman MCP Connection
1. New → MCP
2. Paste server URL (e.g. `https://xxxx.ngrok-free.app/sse`)
3. Connect → Load Capabilities → Tools tab → Run

### ngrok Setup
```python
from pyngrok import ngrok
ngrok.set_auth_token("your_token")
public_url = ngrok.connect(8000).public_url
# Your server is at: {public_url}/sse
```

### Key Concepts
| Term | Meaning |
|------|--------|
| **MCP** | Model Context Protocol — open standard for AI-tool communication |
| **MCP Server** | Exposes tools, resources, and prompts to AI agents |
| **MCP Client** | Connects to servers and calls tools (Postman, Claude, Copilot) |
| **Tool** | An action the AI can perform (like an API endpoint) |
| **SSE** | Server-Sent Events — the transport used for network MCP connections |
| **FastMCP** | Python framework for building MCP servers |
| **Docstring** | The function description — this IS the tool’s documentation for AI |